Reducers usados

In [ ]:
def unicos(atual : list, novo :list)-> list:
    return list(dict.fromkeys((atual or [])+(novo or [])))
    
def mesclar(atual : dict, novo :dict)-> dict:
    return {
        **(atual or {}),
        **(novo or {})}

def maior(atual:float, novo:float)->float:
    return max((atual or 0),(novo or 0))

Imagine que temos este estado:

In [4]:
estado = {
    "fontes": ["doc1", "doc2"],
    "pontuacao": 0.75,
    "metadados": {"origem": "documentos"}
}

E recebemos uma atualização:

In [5]:
atualizacao = {
    "fontes": ["doc2", "site_1"],
    "pontuacao": 0.92,
    "metadados": {"origem": "web", "total": 3}
}

Nosso objetivo é combinar os dois dicionários, mas respeitando uma regra diferente para cada campo.

1 - Poderíamos fazer isso manualmente:

In [8]:
novo_estado = {
    "fontes": unicos(
        estado["fontes"],
        atualizacao["fontes"]
    ),

    "pontuacao": maior(
        estado["pontuacao"],
        atualizacao["pontuacao"]
    ),

    "metadados": mesclar(
        estado["metadados"],
        atualizacao["metadados"]
    )
}

print(novo_estado)

{'fontes': ['doc1', 'doc2', 'site_1'], 'pontuacao': 0.92, 'metadados': {'origem': 'web', 'total': 3}}


2 - Utilizando funções como valores
funções são objetos de primeira classe no Python?

Isso significa que podemos:
- Armazenar funções em variáveis.
- Armazenar funções em listas e dicionários.
- Passar funções como argumentos.
- Retornar funções dentro de outras funções.
Vamos explorar a segunda possibilidade.

Podemos criar um dicionário assim:

In [9]:
reducers = {
    "fontes": unicos,
    "pontuacao": maior,
    "metadados": mesclar
}

Não escrevemos o '()' nas funções, estamos escrevendo uma referencia para elas e não as executando.

Para executar podemos fazer:

In [10]:
funcao = reducers['fontes']
resultado = funcao(["doc1", "doc2"],["doc2", "site_1"])
print(resultado)

['doc1', 'doc2', 'site_1']


Com isso, podemos automatizar o uso dos reducers num dict

In [13]:
for k in estado.items():
    print(k)

fontes ['doc1', 'doc2']
pontuacao 0.75
metadados {'origem': 'documentos'}


In [15]:
estado_novo = {}
for k in estado:
    reducer = reducers[k]
    estado_novo[k] = reducer(
        estado[k],atualizacao[k]
    )

print(estado_novo)

{'fontes': ['doc1', 'doc2', 'site_1'], 'pontuacao': 0.92, 'metadados': {'origem': 'web', 'total': 3}}


#### Exercício 1 — Atualização automática
Crie uma função chamada atualizar_estado(estado, atualizacao, reducers). Ela deve percorrer as chaves do estado, recuperar o reducer correspondente e construir um novo dicionário com os resultados.

In [18]:
def atualizarEstado(state: dict, atualizacao:dict, reducers: dict):
    estado_novo = {}
    for chave in state:
        reducer = reducers[chave]

        estado_novo[chave] = reducer(state[chave],atualizacao[chave])
    
    return estado_novo


In [19]:
print(atualizarEstado(estado,atualizacao,reducers))

{'fontes': ['doc1', 'doc2', 'site_1'], 'pontuacao': 0.92, 'metadados': {'origem': 'web', 'total': 3}}


#### Exercício 2 — Campo opcional
Adapte sua função para que ela também funcione quando uma chave estiver ausente em atualizacao. Nesse caso, o valor atual deve ser preservado.

In [25]:
def atualizarEstado2(state: dict, atualizacao:dict, reducers: dict):
    estado_novo = {}
    for chave in state:
        reducer = reducers[chave]

        estado_novo[chave] = reducer(state[chave],atualizacao.get(chave,state[chave]))
    
    return estado_novo

O que atualizacao.get(chave,state[chave]) faz ?
- Se a chave existir, retorna seu valor.
- Se não existir, retorna o valor padrão.

In [26]:
print(atualizarEstado2(estado,atualizacao,reducers))

{'fontes': ['doc1', 'doc2', 'site_1'], 'pontuacao': 0.92, 'metadados': {'origem': 'web', 'total': 3}}
